# Stages 2 and 3 - Source corruption generation

The summary is never touched. Only the source document is edited.

| Pipeline | Varies | Holds fixed | Effect on the untouched summary |
|---|---|---|---|
| Deletion | N = 1, 2, 3 | operation | claim becomes **unsupported** |
| Alteration | Substitute, Coarsen | N = 1 | claim becomes **contradicted** |

Targets are chosen in **code**, not by the model, so the gold span is exact and the category balance is controllable. `common.targetable` applies the shared rules: Specialist and Medical Procedure are never targeted, Other is targeted by deletion only, and Symptom is capped at one target per document.

Both generators run over all 50 documents, writing four trimmed CSVs. Verification (`verification.py`) is parked.

In [1]:
import os
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
import requests
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath("source_corruption"))
import b1_deletion
import b2_alteration
from common import FACT_CATEGORIES, parse_rewritten_source, parse_spans

load_dotenv(os.path.abspath("../../.env"))
api_keys = [k for k in (os.getenv("OPENROUTER_API_KEY_NEW"), os.getenv("OPENROUTER_API_KEY")) if k]
if not api_keys:
    raise ValueError("No API key found! Check the .env file at the repository root.")

GENERATORS = {"luna": "openai/gpt-5.6-luna"}
INPUT_TAGS = "facts_full_tag_columns.csv"
MAX_WORKERS = 4

print(f"{len(api_keys)} key(s) loaded. Generators: {list(GENERATORS.values())}")

2 key(s) loaded. Generators: ['openai/gpt-5.6-luna']


In [2]:
# Reconstitute typed facts from the tag-column layout.
tags = pd.read_csv(INPUT_TAGS)

documents = []
for _, row in tags.iterrows():
    facts = []
    for category in FACT_CATEGORIES:
        cell = row.get(category)
        if pd.isna(cell) or not str(cell).strip():
            continue
        for text in str(cell).split(" | "):
            if text.strip():
                facts.append((len(facts) + 1, category, text.strip()))
    if facts:
        documents.append(
            {"id": row["id"], "split": row.get("split", ""), "source": row["source"],
             "summary": row["summary"], "facts": facts}
        )

jobs = []
for doc in documents:
    built = b1_deletion.build_jobs(doc["id"], doc["source"], doc["facts"])
    built += b2_alteration.build_jobs(doc["id"], doc["source"], doc["facts"])
    for job in built:
        for short, slug in GENERATORS.items():
            variant = job.get("method_name") or f"Delete_N{job['n']}"
            jobs.append({**job, "gen": short, "model": slug,
                         "job_key": f"{short}|{job['doc_id']}|{variant}",
                         "split": doc["split"],
                         "source_original": doc["source"], "summary": doc["summary"]})

plan = pd.DataFrame([{"gen": j["gen"], "pipeline": j["pipeline"],
                      "variant": j.get("method_name", f"Delete N={j['n']}")} for j in jobs])
print(f"{len(documents)} documents -> {len(jobs)} API calls\n")


2331 documents -> 9472 API calls



In [3]:
key_lock = threading.Lock()
active_key_index = 0


def call_model(prompt, model_name, max_retries=3):
    """Send one prompt to OpenRouter. Rotates keys when one runs out of credit."""
    global active_key_index
    # max_tokens bounds a runaway generation. The rewritten document is at
    # most ~264 tokens (longest source is 528 chars) plus two span lines.
    payload = {"model": model_name, "messages": [{"role": "user", "content": prompt}],
               "temperature": 0.0, "max_tokens": 2048,
               "reasoning": {"enabled": False}}

    for attempt in range(max_retries):
        for _ in range(len(api_keys)):
            with key_lock:
                key = api_keys[active_key_index]
            try:
                response = requests.post(
                    "https://openrouter.ai/api/v1/chat/completions",
                    headers={"Authorization": f"Bearer {key}", "Content-Type": "application/json"},
                    json=payload, timeout=180,
                )
            except requests.RequestException:
                break
            if response.status_code == 200:
                return response.json()["choices"][0]["message"]["content"]
            # 403 is what OpenRouter returns for "Key limit exceeded".
            if response.status_code in (401, 402, 403, 429):
                with key_lock:
                    active_key_index = (active_key_index + 1) % len(api_keys)
                continue
            break
        time.sleep(2 * (attempt + 1))
    return None


def run_job(job):
    raw = call_model(job["prompt"], job["model"])
    corrupted = parse_rewritten_source(raw) if raw else None
    original_span, new_span = parse_spans(raw) if raw else (None, None)
    return {
        "gen": job["gen"],
        "pipeline": job["pipeline"],
        "doc_id": job["doc_id"],
        "split": job.get("split", ""),
        "method": job.get("method_name", "Delete"),
        "n": job["n"],
        "target_categories": " | ".join(job["target_categories"]),
        "target_facts": " | ".join(t[2] for t in job["targets"]),
        "retained_facts": " | ".join(t[2] for t in job["retained"]),
        "original_span": original_span,
        "new_span": new_span,
        "source_original": job["source_original"],
        "source_corrupted": corrupted,
        "summary": job["summary"],
        "ok": corrupted is not None,
    }


print("Ready.")

Ready.


In [4]:
# PRE-FLIGHT. Two calls, about $0.003, before committing to the full run.
# If either fails to parse, the run below must not be started.
assert all(str(d["source"]).strip() for d in documents), "a document has an empty source"
assert len({j["job_key"] for j in jobs}) == len(jobs), "duplicate job_key - jobs would be double-paid"

smoke_ok = True
for short, slug in GENERATORS.items():
    probe = next(j for j in jobs if j["gen"] == short)
    raw = call_model(probe["prompt"], probe["model"])
    parsed = parse_rewritten_source(raw) if raw else None
    span = parse_spans(raw) if raw else (None, None)
    status = "OK" if parsed else "FAILED"
    if not parsed:
        smoke_ok = False
    print(f"{status:>6}  {slug:<28} parsed={bool(parsed)} span={span[1] if span else None}")

if not smoke_ok:
    raise RuntimeError("Pre-flight failed. Fix the prompt or model before running 406 calls.")

print("Pre-flight passed. Safe to run the cell below.")

    OK  openai/gpt-5.6-luna          parsed=True span=NONE
Pre-flight passed. Safe to run the cell below.


In [5]:
BACKUP_FILE = "backup_full_generation.csv"
CHECKPOINT_EVERY = 20

# Resume: anything already in the backup is not re-requested. The run can be
# killed at any point and restarted without losing or paying for work twice.
done = {}
if os.path.exists(BACKUP_FILE):
    prior = pd.read_csv(BACKUP_FILE)
    done = {r["job_key"]: r for _, r in prior.iterrows()}
    print(f"Found {BACKUP_FILE}: resuming, {len(done)} job(s) already complete.")
else:
    print("No backup found - starting fresh.")

pending = [j for j in jobs if j["job_key"] not in done]
print(f"{len(pending)} of {len(jobs)} jobs still to run.")
print("")

start = time.time()
results = [done[j["job_key"]].to_dict() for j in jobs if j["job_key"] in done]
lock = threading.Lock()


def checkpoint():
    pd.DataFrame(results).to_csv(BACKUP_FILE, index=False, encoding="utf-8-sig")


if pending:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        for finished, record in enumerate(pool.map(run_job, pending), start=1):
            with lock:
                results.append(record)
                if finished % CHECKPOINT_EVERY == 0 or finished == len(pending):
                    checkpoint()
                    ok = sum(bool(r.get("ok")) for r in results)
                    print(f"   --- saved at {finished}/{len(pending)} "
                          f"(total {len(results)}, ok={ok}, {time.time() - start:.0f}s) ---")

checkpoint()
out = pd.DataFrame(results)
out["ok"] = out["ok"].astype(bool)
print("")
print(f"{len(out)} jobs, {int(out['ok'].sum())} succeeded, {int((~out['ok']).sum())} failed")

No backup found - starting fresh.
9472 of 9472 jobs still to run.



   --- saved at 20/9472 (total 20, ok=20, 14s) ---


   --- saved at 40/9472 (total 40, ok=40, 26s) ---


   --- saved at 60/9472 (total 60, ok=60, 57s) ---
   --- saved at 80/9472 (total 80, ok=80, 57s) ---


   --- saved at 100/9472 (total 100, ok=99, 70s) ---


...

   --- saved at 9420/9472 (total 9420, ok=9369, 5760s) ---


   --- saved at 9440/9472 (total 9440, ok=9389, 5772s) ---


   --- saved at 9460/9472 (total 9460, ok=9409, 5784s) ---


   --- saved at 9472/9472 (total 9472, ok=9421, 5791s) ---



9472 jobs, 9421 succeeded, 51 failed


In [6]:
# Four trimmed CSVs. Generator and pipeline live in the filename, not a column.
# Failed rows are dropped rather than carried as blanks; the count is printed.
ALTERATION_COLS = ["doc_id", "split", "method", "target_categories", "target_facts", "retained_facts",
                   "original_span", "new_span", "source_original", "source_corrupted", "summary"]
DELETION_COLS = ["doc_id", "split", "method", "n", "target_categories", "target_facts", "retained_facts",
                 "original_span", "source_original", "source_corrupted", "summary"]

print(f"{'file':<40} {'rows':>5} {'dropped':>8} {'cols':>5}")
print("-" * 62)
for short in GENERATORS:
    for pipeline, cols in (("deletion", DELETION_COLS), ("alteration", ALTERATION_COLS)):
        subset = out[(out["gen"] == short) & (out["pipeline"] == pipeline)]
        dropped = int((~subset["ok"]).sum())
        frame = subset[subset["ok"]][cols]
        path = f"full_{short}_{pipeline}.csv"
        frame.to_csv(path, index=False, encoding="utf-8-sig")
        print(f"{path:<40} {len(frame):>5} {dropped:>8} {len(cols):>5}")

file                                      rows  dropped  cols
--------------------------------------------------------------
full_luna_deletion.csv                    5641       49    11


full_luna_alteration.csv                  3780        2    11


In [7]:
# Yield per variant, and the silent-failure check: a source returned unchanged.
ok = out[out["ok"]].copy()
ok["variant"] = ok.apply(
    lambda r: r["method"] if r["pipeline"] == "alteration" else f"Delete N={r['n']}", axis=1
)
table = (out.assign(variant=out.apply(
            lambda r: r["method"] if r["pipeline"] == "alteration" else f"Delete N={r['n']}", axis=1))
         .groupby(["gen", "variant"]).agg(jobs=("ok", "size"), ok=("ok", "sum")))
table["yield%"] = (100 * table["ok"] / table["jobs"]).round(0)
print(table.to_string())

unchanged = ok[ok["source_corrupted"].str.strip() == ok["source_original"].str.strip()]
print(f"\nSources returned unchanged (silent failures): {len(unchanged)}")
print(f"Items missing a gold span: {int(ok['original_span'].isna().sum() - (ok['pipeline'] == 'alteration').sum() * 0)}")

                 jobs    ok  yield%
gen  variant                       
luna Coarsen     1472  1471   100.0
     Delete N=1  2282  2255    99.0
     Delete N=2  1956  1945    99.0
     Delete N=3  1452  1441    99.0
     Substitute  2310  2309   100.0

Sources returned unchanged (silent failures): 30
Items missing a gold span: 19
